# pycares smoke test

`pycares` is a Python binding to the c-ares asynchronous DNS resolver. Live DNS resolution needs network access, so this notebook exercises only the **offline** API surface: channel construction/config, the query-type + flag constants, the errno table, and the pure `parse_name` helper.

In [ ]:
# Example 1: construct a Channel with default + custom configuration (no queries issued).
import pycares

default = pycares.Channel()
print(f"default servers: {default.servers}")
print(f"default timeout: {default.timeout()}")
assert isinstance(default.servers, list)
default.close()

custom = pycares.Channel(servers=['8.8.8.8', '1.1.1.1'], timeout=3, tries=2,
                         flags=pycares.ARES_FLAG_NOCHECKRESP)
print(f"custom servers: {custom.servers}")
# .servers may render each entry with a ':53' port suffix depending on the c-ares version.
for ip in ('8.8.8.8', '1.1.1.1'):
    assert any(ip in s for s in custom.servers), f"{ip} not among {custom.servers}"
custom.close()

In [ ]:
# Example 2: the query-type and flag constant tables are populated.
import pycares

query_types = {n: getattr(pycares, n) for n in dir(pycares) if n.startswith('QUERY_TYPE_')}
flags = {n: getattr(pycares, n) for n in dir(pycares) if n.startswith('ARES_FLAG_')}
print(f"{len(query_types)} query-type constants, {len(flags)} flag constants")

# Canonical record types are present with integer codes.
for name in ('QUERY_TYPE_A', 'QUERY_TYPE_AAAA', 'QUERY_TYPE_MX', 'QUERY_TYPE_TXT', 'QUERY_TYPE_PTR'):
    assert name in query_types and isinstance(query_types[name], int)
assert 'ARES_FLAG_NOCHECKRESP' in flags
print(f"QUERY_TYPE_A={pycares.QUERY_TYPE_A}, QUERY_TYPE_AAAA={pycares.QUERY_TYPE_AAAA}")

In [ ]:
# Example 3: the errno table maps codes to human-readable strings.
import pycares
from pycares import errno

codes = [n for n in dir(errno) if n.startswith('ARES_E')]
print(f"{len(codes)} error codes; sample: {codes[:5]}")
assert codes, "expected a populated errno table"

# strerror turns a code into a message.
msg = errno.strerror(errno.ARES_ENODATA)
print(f"strerror(ARES_ENODATA) = {msg!r}")
assert isinstance(msg, str) and msg

# AresError is the package's exception type.
assert issubclass(pycares.AresError, Exception)

In [ ]:
# Example 4: parse_name is a pure (offline) helper — IDNA/punycode-encode a name to bytes.
import pycares

# Plain ASCII names pass through unchanged (as bytes).
ascii_name = pycares.parse_name('www.example.com')
print(f"parse_name('www.example.com') = {ascii_name!r}")
assert ascii_name == b'www.example.com'

# Unicode names are IDNA-encoded to their ASCII-compatible (punycode) form.
idna = pycares.parse_name('münchen.de')
print(f"parse_name('münchen.de') = {idna!r}")
assert idna == b'xn--mnchen-3ya.de'

# It is idempotent on already-encoded bytes.
assert pycares.parse_name(idna) == idna

In [ ]:
# Example 5: pycares exposes a typed record-data class per DNS record type.
import pycares

record_types = [n for n in dir(pycares) if n.endswith('RecordData')]
print(f"{len(record_types)} record-data classes")
print(record_types[:8])

for name in ('ARecordData', 'AAAARecordData', 'MXRecordData', 'TXTRecordData', 'CNAMERecordData'):
    assert hasattr(pycares, name), f"missing record class {name}"
# The high-level result wrappers are exported too.
for name in ('DNSResult', 'DNSRecord', 'AddrInfoResult'):
    assert hasattr(pycares, name), f"missing result wrapper {name}"

# AresError round-trips through raise/except.
try:
    raise pycares.AresError('synthetic failure')
except pycares.AresError as e:
    caught = str(e)
assert caught == 'synthetic failure'
print("pycares smoke test PASS")